In [1]:
import glob
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from keras.utils import img_to_array, load_img
from sklearn.model_selection import train_test_split


In [2]:
# Update this path if your dataset lives somewhere else.
IMAGES_PATH = "/Users/amiteshwarsingh/Documents/AI_ML_DL/classification_pet_faces/data/raw/oxford_IIIT_pet_dataset/images"
IMG_SIZE = (224, 224)
TEST_SIZE = 0.2
VAL_SIZE = 0.25
RANDOM_STATE_TEST = 42
RANDOM_STATE_VAL = 1

def image_name_to_label(name):
    return " ".join(os.path.splitext(name.lower())[0].rsplit("_", 1)[0].split("_"))

def assign_label_code(images_label):
    return {label: i for i, label in enumerate(images_label)}

def load_dataset_paths(images_path):
    return [os.path.basename(file) for file in glob.glob(os.path.join(images_path, "*.jpg"))]

images_name = load_dataset_paths(IMAGES_PATH)
images_label = sorted(set(image_name_to_label(name) for name in images_name))
label_to_code = assign_label_code(images_label)

len(images_name), len(images_label)


(7390, 37)

In [3]:
def features_and_labels(images_name):
    features = []
    labels = []

    for name in images_name:
        label = image_name_to_label(name)
        label_code = label_to_code.get(label.lower())
        if label_code is None:
            continue

        img = load_img(os.path.join(IMAGES_PATH, name), target_size=IMG_SIZE)
        img = img_to_array(img, dtype="uint8")
        features.append(img)
        labels.append(label_code)

    return np.array(features), np.array(labels)

features_array, labels_array = features_and_labels(images_name)
features_array.shape, labels_array.shape


((7390, 224, 224, 3), (7390,))

In [4]:
# Train / validation / test split
X_train, X_test, y_train, y_test = train_test_split(
    features_array,
    labels_array,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE_TEST,
    stratify=labels_array,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=VAL_SIZE,
    random_state=RANDOM_STATE_VAL,
    stratify=y_train,
)

# Normalize pixels to [0, 1]
X_train = X_train.astype("float32") / 255.0
X_val = X_val.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

X_train.shape, X_val.shape, X_test.shape


((4434, 224, 224, 3), (1478, 224, 224, 3), (1478, 224, 224, 3))

In [5]:
# Augmentation layer you can tweak during experiments.
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),
    data_augmentation,
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(37, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 186624)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    23,888,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 37)             │         4,773 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,912,165 (91.22 MB)

 Trainable params: 23,912,165 (91.22 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[early_stopping],
)


In [ ]:
epochs_range = range(1, len(history.history["accuracy"]) + 1)
plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history.history["accuracy"], label="Training accuracy")
plt.plot(epochs_range, history.history["val_accuracy"], label="Validation accuracy")
plt.legend(loc="lower right")
plt.title("Training and Validation Accuracy")

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history.history["loss"], label="Training loss")
plt.plot(epochs_range, history.history["val_loss"], label="Validation loss")
plt.legend(loc="upper right")
plt.title("Training and Validation Loss")

plt.tight_layout()
plt.show()


In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

history_df = pd.DataFrame(history.history)
history_df.insert(0, "phase", [f"epoch_{i + 1}" for i in range(len(history_df))])

test_df = pd.DataFrame([
    {
        "phase": "test",
        "loss": test_loss,
        "accuracy": test_accuracy,
        "val_loss": np.nan,
        "val_accuracy": np.nan,
    }
])

results_df = pd.concat([history_df, test_df], ignore_index=True)
results_df


In [ ]:
output_dir = os.path.dirname(__file__) if "__file__" in globals() else os.getcwd()

plt.figure(figsize=(15, 6))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history.history["accuracy"], label="Training accuracy")
plt.plot(epochs_range, history.history["val_accuracy"], label="Validation accuracy")
plt.legend(loc="lower right")
plt.title("Training and Validation Accuracy")

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history.history["loss"], label="Training loss")
plt.plot(epochs_range, history.history["val_loss"], label="Validation loss")
plt.legend(loc="upper right")
plt.title("Training and Validation Loss")

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "basic_model_training_curves.png"))
plt.close()

results_path = os.path.join(output_dir, "training_validation_test_results.csv")
results_df.to_csv(results_path, index=False)

print(f"Saved plot to: {os.path.join(output_dir, 'basic_model_training_curves.png')}")
print(f"Saved results to: {results_path}")
